# PyTorch Dataset and DataLoader 详细讲解
## 一、前言
在PyTorch中，手动喂入数据的方式在处理大规模数据集时存在效率低、不支持批量训练等问题，而`Dataset`和`DataLoader`是PyTorch提供的核心数据加载工具，能够实现数据的高效读取、批量处理、打乱顺序、多进程加载等功能，是深度学习训练流程中数据处理环节的关键组件。

本文将从手动喂数的弊端出发，讲解核心术语、`Dataset`和`DataLoader`的定义与使用、Windows系统下的注意事项，同时结合实战案例和官方内置数据集进行说明，最后给出实战练习方向。

## 二、手动数据喂入的弊端
传统的手动加载数据方式直接读取全部数据并转换为Tensor，训练时一次性使用所有数据，示例代码如下：
```python
import numpy as np
import torch

# 手动加载糖尿病数据集
xy = np.loadtxt('diabetes.csv.gz', delimiter=',', dtype=np.float32)
x_data = torch.from_numpy(xy[:,:-1])
y_data = torch.from_numpy(xy[:, [-1]])

# 训练循环（一次性使用所有数据）
for epoch in range(100):
    y_pred = model(x_data)  # 前向传播
    loss = criterion(y_pred, y_data)  # 计算损失
    print(epoch, loss.item())
    optimizer.zero_grad()  # 梯度清零
    loss.backward()  # 反向传播
    optimizer.step()  # 参数更新
```
**核心弊端**：
1. 一次性加载全部数据，占用大量内存，无法处理超大规模数据集；
2. 不支持批量训练（Batch Training），模型训练效果差、收敛慢；
3. 无数据打乱功能，易导致模型过拟合；
4. 不支持多进程并行加载，数据读取效率低。

## 三、核心术语：Epoch、Batch-Size、Iterations
在使用`Dataset`和`DataLoader`时，必须理解三个核心训练术语，三者共同构成批量训练的循环逻辑：
```python
# 训练循环的通用结构
for epoch in range(training_epochs):  # 遍历所有轮次
    for i in range(total_batch):      # 遍历所有批次
        # 批次数据处理、前向传播、反向传播、参数更新
```

### 1. Epoch（轮次）
**定义**：对**所有训练样本**完成一次前向传播和一次反向传播，即模型把整个训练集学一遍。
- 示例：训练集有1000个样本，完成1000个样本的前向+反向传播，就是1个Epoch。

### 2. Batch-Size（批次大小）
**定义**：一次前向传播和反向传播中使用的**训练样本数量**，即每次喂给模型的样本数。
- 示例：Batch-Size=32，表示每次取32个样本训练模型。

### 3. Iterations（迭代次数）
**定义**：完成一个Epoch所需的**批次传播次数**，即遍历整个训练集需要的批次数量。
- 计算公式：$Iterations = \lceil \frac{总样本数}{Batch-Size} \rceil$（向上取整）
- 示例：1000个样本，Batch-Size=32，迭代次数=32（31*32=992，最后一批8个样本）。

## 四、Dataset 抽象类：自定义数据集的基础
### 1. 核心作用
`Dataset`是PyTorch中**抽象的数据集基类**，所有自定义数据集都需要继承该类并实现其**三个核心魔法方法**，用于定义数据的读取、索引和长度获取规则，让PyTorch能够统一处理不同格式的数据集（如CSV、图片、文本等）。

### 2. 导入方式
```python
from torch.utils.data import Dataset
```

### 3. 必须实现的三个方法
自定义数据集类的模板如下，每个方法都有固定的功能和实现要求：
```python
import torch
from torch.utils.data import Dataset

class CustomDataset(Dataset):
    def __init__(self):
        """初始化方法：加载数据集、预处理数据（如读取文件、转换格式）"""
        pass
    
    def __getitem__(self, index):
        """
        索引方法：通过索引index获取对应位置的样本
        :param index: 样本的索引（int类型）
        :return: 单个样本（通常为(input, label)元组）
        """
        pass
    
    def __len__(self):
        """长度方法：返回数据集的总样本数"""
        pass
```

#### （1）`__init__(self)`
- 执行时机：创建数据集对象时自动调用；
- 核心操作：读取数据文件（如CSV、图片文件夹）、数据预处理（如归一化、类型转换）、初始化样本和标签的存储容器（如Tensor、列表）。

#### （2）`__getitem__(self, index)`
- 执行时机：使用`dataset[index]`索引样本时自动调用；
- 核心要求：必须返回**单个样本的输入和标签**（元组形式），是实现批量加载、数据打乱的基础；
- 核心意义：让自定义数据集支持**索引访问**，与PyTorch的DataLoader兼容。

#### （3）`__len__(self)`
- 执行时机：使用`len(dataset)`时自动调用；
- 核心要求：必须返回**整数类型的总样本数**；
- 核心意义：让DataLoader能够计算迭代次数，确定遍历数据集的终止条件。

### 4. 实战案例：糖尿病数据集（DiabetesDataset）
以糖尿病数据集（CSV格式）为例，实现自定义`Dataset`，完整代码如下：
```python
import numpy as np
import torch
from torch.utils.data import Dataset

class DiabetesDataset(Dataset):
    def __init__(self, filepath):
        """加载糖尿病数据集并完成初始化"""
        # 读取CSV文件，按逗号分隔，转换为float32类型
        xy = np.loadtxt(filepath, delimiter=',', dtype=np.float32)
        self.len = xy.shape[0]  # 总样本数，赋值给实例变量
        self.x_data = torch.from_numpy(xy[:, :-1])  # 输入特征（所有行，除最后一列）
        self.y_data = torch.from_numpy(xy[:, [-1]]) # 标签（所有行，最后一列，保持二维）
    
    def __getitem__(self, index):
        """通过索引获取单个样本"""
        return self.x_data[index], self.y_data[index]
    
    def __len__(self):
        """返回数据集总样本数"""
        return self.len

# 创建数据集对象
dataset = DiabetesDataset('diabetes.csv.gz')
```

## 五、DataLoader 类：数据集的批量加载器
### 1. 核心作用
`DataLoader`是PyTorch提供的**数据加载器**，基于自定义的`Dataset`对象，实现**批量采样、数据打乱、多进程并行加载**等功能，将数据集封装为可迭代的批次数据生成器，训练时可直接遍历获取批次样本，解决手动喂数的所有弊端。

### 2. 导入方式
```python
from torch.utils.data import DataLoader
```

### 3. 初始化参数
核心参数说明（常用且重要），其余参数可参考PyTorch官方文档：
| 参数名 | 类型 | 核心作用 | 示例 |
|--------|------|----------|------|
| `dataset` | Dataset对象 | 传入自定义/官方的Dataset实例，是数据来源 | `dataset=DiabetesDataset('diabetes.csv.gz')` |
| `batch_size` | int | 批次大小，即每次返回的样本数 | `batch_size=32` |
| `shuffle` | bool | 是否在每个Epoch前打乱数据顺序，防止过拟合 | `shuffle=True`（训练集）/`False`（测试集） |
| `num_workers` | int | 用于数据加载的**多进程数**，0表示主进程加载 | `num_workers=2` |
| `drop_last` | bool | 是否丢弃最后一批不足`batch_size`的样本 | `drop_last=False`（默认，保留最后一批） |

### 4. 初始化示例
基于上述的`DiabetesDataset`创建`DataLoader`：
```python
train_loader = DataLoader(
    dataset=dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2
)
```

### 5. 核心特性
1. **可迭代性**：训练时可通过`enumerate`遍历，每次返回**批次索引**和**批次数据（inputs, labels）**；
2. **自动打乱**：若`shuffle=True`，每个Epoch开始前会自动打乱数据顺序，保证训练的随机性；
3. **多进程加载**：`num_workers>0`时，开启多进程并行读取数据，提升数据加载效率（适合大规模数据集）；
4. **动态批次**：自动处理最后一批样本数不足的情况，无需手动补全。

## 六、Windows 系统下的 num_workers 注意事项
Windows系统的多进程实现基于`spawn`（Linux/macOS基于`fork`），直接使用`num_workers>0`会导致**进程启动错误**，抛出如下异常：
```
RuntimeError: An attempt has been made to start a new process before the current process has finished its bootstrapping phase.
```

### 解决方法
将**数据加载和训练的循环代码**包裹在`if __name__ == '__main__':`语句中，防止多进程重复执行主模块代码，示例：
```python
if __name__ == '__main__':
    # 创建数据集和DataLoader
    dataset = DiabetesDataset('diabetes.csv.gz')
    train_loader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=2)
    # 训练循环
    for epoch in range(100):
        for i, data in enumerate(train_loader, 0):
            # 后续处理逻辑
            pass
```
> 注：`freeze_support()`可省略，仅在将代码打包为可执行文件时需要添加。

## 七、Dataset + DataLoader 实战：糖尿病分类模型
结合自定义数据集、数据加载器和PyTorch模型搭建，实现完整的糖尿病分类训练流程，**含完整可运行代码**：
```python
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

# 1. 定义自定义数据集
class DiabetesDataset(Dataset):
    def __init__(self, filepath):
        xy = np.loadtxt(filepath, delimiter=',', dtype=np.float32)
        self.len = xy.shape[0]
        self.x_data = torch.from_numpy(xy[:, :-1])
        self.y_data = torch.from_numpy(xy[:, [-1]])
    
    def __getitem__(self, index):
        return self.x_data[index], self.y_data[index]
    
    def __len__(self):
        return self.len

# 2. 定义神经网络模型
class Model(torch.nn.Module):
    def __init__(self):
        super(Model, self).__init__()
        # 8输入→6隐藏→4隐藏→1输出（糖尿病为二分类，输出0/1）
        self.linear1 = torch.nn.Linear(8, 6)
        self.linear2 = torch.nn.Linear(6, 4)
        self.linear3 = torch.nn.Linear(4, 1)
        self.sigmoid = torch.nn.Sigmoid()  # Sigmoid激活，将输出映射到0-1
    
    def forward(self, x):
        x = self.sigmoid(self.linear1(x))
        x = self.sigmoid(self.linear2(x))
        x = self.sigmoid(self.linear3(x))
        return x

# 3. 主程序：数据加载+模型训练
if __name__ == '__main__':
    # 初始化数据集和DataLoader
    dataset = DiabetesDataset('diabetes.csv.gz')
    train_loader = DataLoader(
        dataset=dataset,
        batch_size=32,
        shuffle=True,
        num_workers=2
    )
    # 初始化模型、损失函数、优化器
    model = Model()
    criterion = torch.nn.BCELoss(size_average=True)  # 二分类用BCELoss
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01)  # 随机梯度下降
    
    # 4. 训练循环（Epoch + Iteration）
    for epoch in range(100):  # 100个Epoch
        for i, data in enumerate(train_loader, 0):  # 遍历所有批次，索引从0开始
            # （1）准备批次数据：解包为输入和标签
            inputs, labels = data
            # （2）前向传播：计算预测值
            y_pred = model(inputs)
            # （3）计算损失
            loss = criterion(y_pred, labels)
            # （4）反向传播：梯度清零→计算梯度
            optimizer.zero_grad()
            loss.backward()
            # （5）参数更新
            optimizer.step()
            # 打印训练信息
            print(f'Epoch: {epoch}, Iteration: {i}, Loss: {loss.item():.4f}')
```

### 核心训练流程
PyTorch深度学习的通用训练流程，也是`Dataset+DataLoader`的标准使用流程：
1. **准备数据集**：通过`Dataset`自定义，`DataLoader`封装为批次迭代器；
2. **设计模型**：继承`torch.nn.Module`，定义网络结构和前向传播；
3. **构造损失函数和优化器**：使用PyTorch内置API（如BCELoss、SGD）；
4. **训练循环**：嵌套遍历Epoch和Iteration，执行**前向传播→计算损失→反向传播→参数更新**。

## 八、PyTorch 内置数据集：torchvision.datasets
PyTorch的`torchvision.datasets`模块提供了**大量经典的内置数据集**（如MNIST、CIFAR10、Fashion-MNIST等），这些数据集都已实现`Dataset`的核心方法（`__getitem__`、`__len__`），可直接与`DataLoader`配合使用，无需手动实现自定义数据集。

### 1. 内置数据集的共性
- 均为`Dataset`的子类，支持直接传入`DataLoader`；
- 提供`download`参数，若本地无数据集，自动从官网下载；
- 提供`transform`参数，支持数据预处理（如转换为Tensor、归一化）；
- 分训练集/测试集：通过`train=True/False`区分。

### 2. 实战案例：MNIST 手写数字数据集
MNIST是经典的手写数字分类数据集（0-9），使用`torchvision.datasets`加载并配合`DataLoader`的示例：
```python
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision import datasets

# 数据预处理：将PIL图片转换为Tensor（自动归一化到0-1）
transform = transforms.ToTensor()

# 加载训练集和测试集
train_dataset = datasets.MNIST(
    root='../dataset/mnist',  # 数据集保存路径
    train=True,               # 训练集
    transform=transform,      # 数据预处理
    download=True             # 本地无则下载
)
test_dataset = datasets.MNIST(
    root='../dataset/mnist',
    train=False,              # 测试集
    transform=transform,
    download=True
)

# 封装为DataLoader
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=32,
    shuffle=True  # 训练集打乱
)
test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=32,
    shuffle=False  # 测试集不打乱
)

# 遍历批次数据
for batch_idx, (inputs, target) in enumerate(train_loader):
    print(f'批次索引：{batch_idx}，输入形状：{inputs.shape}，标签形状：{target.shape}')
    break
```
> 输出：`批次索引：0，输入形状：torch.Size([32, 1, 28, 28])，标签形状：torch.Size([32])`
> 说明：MNIST样本为28×28的单通道图片，`inputs.shape`为`[batch_size, channel, height, width]`。

### 3. 常用内置数据集
| 数据集名称 | 任务类型 | 样本特点 |
|------------|----------|----------|
| MNIST | 手写数字分类（0-9） | 28×28单通道灰度图，6万训练/1万测试 |
| Fashion-MNIST | 服装分类（10类） | 28×28单通道灰度图，6万训练/1万测试 |
| CIFAR10/CIFAR100 | 图像分类 | 32×32三通道彩色图，CIFAR10为10类，CIFAR100为100类 |
| COCO | 目标检测/分割/关键点检测 | 大规模场景图像，含丰富的标注信息 |
| ImageFolder | 自定义图片分类 | 按文件夹名分类，适合自己的图片数据集 |

## 九、实战练习：Titanic 泰坦尼克号数据集
### 练习任务
1. 从Kaggle下载泰坦尼克号数据集（https://www.kaggle.com/c/titanic/data），包含`train.csv`（训练集，891×12）和`test.csv`（测试集，418×11）；
2. 继承`Dataset`实现**TitanicDataset**，处理数据（如缺失值填充、类别特征编码、数值特征归一化）；
3. 使用`DataLoader`封装训练集，设置合适的`batch_size`和`shuffle`；
4. 搭建简单的神经网络分类器，预测乘客的生存概率（二分类任务）；
5. 训练模型并验证效果。

### 核心数据处理要点
1. 缺失值：Age、Cabin、Embarked列存在缺失值，需填充（如均值、中位数、众数）；
2. 类别特征：Sex（男/女）、Embarked（登船港口）为类别特征，需转换为数值（如独热编码、标签编码）；
3. 无用特征：PassengerId、Name、Ticket、Cabin列可根据情况删除，减少噪声；
4. 特征归一化：数值特征（Age、Fare、SibSp、Parch）需归一化到同一尺度，提升模型收敛速度。

## 十、总结
1. `Dataset`是PyTorch的**抽象数据集基类**，自定义数据集必须继承并实现`__init__`、`__getitem__`、`__len__`三个方法，用于定义数据的读取、索引和长度；
2. `DataLoader`是**批次数据加载器**，基于`Dataset`实现批量采样、数据打乱、多进程加载，将数据集封装为可迭代的批次生成器；
3. 批量训练的核心是**Epoch（轮次）、Batch-Size（批次大小）、Iterations（迭代次数）**，三者构成嵌套的训练循环；
4. Windows系统使用`num_workers>0`时，需将训练代码包裹在`if __name__ == '__main__':`中，避免进程错误；
5. PyTorch的`torchvision.datasets`提供了大量内置数据集，可直接与`DataLoader`配合使用，无需手动实现；
6. PyTorch深度学习的通用流程：**准备数据集（Dataset+DataLoader）→设计模型→构造损失和优化器→训练循环（前向+反向+更新）**。

`Dataset`和`DataLoader`是PyTorch数据处理的基础，掌握其使用方法是实现高效、可扩展的深度学习训练的关键，后续处理任何自定义数据集（如图片、文本、音频）都可遵循此范式。

In [1]:
import torch
print(torch.xpu.is_available())  # torch.xpu is the API for Intel GPU support

False


/home/meal/AI-agent/venv/lib/python3.12/site-packages/torch/xpu/__init__.py:60: UserWarning: XPU device count is zero! (Triggered internally at /pytorch/c10/xpu/XPUFunctions.cpp:115.)
  return torch._C._xpu_getDeviceCount()


In [ ]:
import torch
import intel_extension_for_pytorch as ipex  # 导入IPEX
torch.set_default_device("xpu")  # 设置默认设备为英特尔XPU（A770）

In [2]:

from torch.utils.data import Dataset,DataLoader
import numpy  as np
class DiabetesDataset(Dataset):
    def __init__(self,inpath) -> None:
        xy = np.loadtxt(inpath, delimiter=',', dtype=np.float32)
        self.length = xy.shape[0]
        self.x_data = torch.from_numpy(xy[:,:-1])
        self.y_data = torch.from_numpy(xy[:,[-1]])
    def __getitem__(self, index):
        return self.x_data[index],self.y_data[index]
    def __len__(self):
        return self.length
dataset = DiabetesDataset('data/diabetes.csv')
trian_loader = DataLoader(dataset=dataset,batch_size=32,shuffle=True,num_workers=2)

class DiabetesModule(torch.nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.linear1 = torch.nn.Linear(8,6)
        self.linear2 = torch.nn.Linear(6,4)
        self.linear3 = torch.nn.Linear(4,1)
        self.sigmoid = torch.nn.Sigmoid()

    def forward(self,x):
        x = self.sigmoid(self.linear1(x)) 
        x = self.sigmoid(self.linear2(x)) 
        x = self.sigmoid(self.linear3(x)) 
        return x

model  = DiabetesModule()
criterion = torch.nn.BCELoss(reduction='sum')
optimizer = torch.optim.SGD(model.parameters(),lr=0.05)

for epoch in range(1000):
    for bitch_index,(inputs,labels) in enumerate(trian_loader):
        y_pred = model(inputs)
        loss = criterion(y_pred,labels)
        print(epoch, bitch_index, loss.item())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    if epoch % 100 == 99:
        print(f"Epoch {epoch}: loss={loss.item():.4f}")
        

0 0 8.190587997436523
1 0 7.453660488128662
2 0 7.085052967071533
3 0 6.904999732971191
4 0 6.817386627197266
5 0 6.774561882019043
6 0 6.753472805023193
7 0 6.742993354797363
8 0 6.737725734710693
9 0 6.735032081604004
10 0 6.733616828918457
11 0 6.732836723327637
12 0 6.73237419128418
13 0 6.732069969177246
14 0 6.731845855712891
15 0 6.731661796569824
16 0 6.731497764587402
17 0 6.7313432693481445
18 0 6.731194496154785
19 0 6.731049537658691
20 0 6.7309041023254395
21 0 6.730759620666504
22 0 6.730615615844727
23 0 6.730472564697266
24 0 6.730329990386963
25 0 6.730186462402344
26 0 6.730043888092041
27 0 6.729900360107422
28 0 6.729757308959961
29 0 6.729614734649658
30 0 6.729471683502197
31 0 6.729328155517578
32 0 6.729184627532959
33 0 6.729041576385498
34 0 6.728899002075195
35 0 6.728754997253418
36 0 6.728611469268799
37 0 6.7284674644470215
38 0 6.728323936462402
39 0 6.728180408477783
40 0 6.728035926818848
41 0 6.727891445159912
42 0 6.727746963500977
43 0 6.727603435516